In [1]:
# @title
!pip install corus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 2.5 MB/s eta 0:00:00


In [2]:
# @title
!pip install pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 30.8 MB/s eta 0:00:00


In [3]:
# @title
!pip install navec

In [4]:
# @title
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

--2025-03-13 11:19:58--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  11.2MB/s    in 2.3s    

2025-03-13 11:20:01 (11.2 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [7]:
# @title
!pip install spacy
!python -m spacy download ru_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 75.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
# @title
!pip install --upgrade pymorphy2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 31.1 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=6b8f0005b0ff0d6184bf0528fbb579f3b9ecfa48aee4ec408cea1e03bd3df800
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt


In [10]:
# @title
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 21.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [11]:
import corus
import pandas as pd
import os
import requests
import re
import pymorphy3
import gensim.models
import spacy
import string
import numpy as np
import navec
import urllib.request

from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec, KeyedVectors
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter
from navec import Navec
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
LENTA_URL = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
LOCAL_FILE = "lenta-ru-news.csv.gz"

random_state=42

def download_file(url, path):
    if not os.path.exists(path):
        print("Файл не найден, скачивание...")
        response = requests.get(url, stream=True)
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024):
                file.write(chunk)
        print("Загрузка завершена.")

def load_lenta_dataset(path=LOCAL_FILE, sample_size=100000):
    download_file(LENTA_URL, path)  # Проверка и скачивание файла
    records = []
    dataset = corus.load_lenta(path)

    for record in tqdm(dataset, desc="Загрузка записей"):
        records.append((record.title, record.text, record.topic))

    df = pd.DataFrame(records, columns=["title", "text", "topic"])

    df = df.sample(n=sample_size, random_state=42)

    topic_counts = df["topic"].value_counts()
    valid_topics = topic_counts[topic_counts >= 1000].index
    df = df[df["topic"].isin(valid_topics)]

    min_class_size = min(Counter(df["topic"]).values())
    df_balanced = df.groupby("topic").apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)

    return df_balanced

data = load_lenta_dataset()

data.head()


Файл не найден, скачивание...
Загрузка завершена.


Загрузка записей: 739351it [01:00, 12164.02it/s]
<ipython-input-12-f7bef3e8beba>:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby("topic").apply(lambda x: x.sample(min_class_size, random_state=42)).reset_index(drop=True)


,title,text,topic
0,Тимошенко усомнилась в правдивости данных о де...,Премьер-министр Украины Юлия Тимошенко 13 мая ...,Бывший СССР
1,Глава Минфина Украины пообещала уйти в отставк...,Министр финансов Украины Наталья Яресько в инт...,Бывший СССР
2,В окрестностях Баку упал учебный самолет,"В озеро Масазыр, находящееся неподалеку от Бак...",Бывший СССР
3,Венгрия отказалась захватывать Закарпатье,Венгрия не стремится захватывать определенные ...,Бывший СССР
4,Захарченко предрек признание Киевом ДНР,Глава самопровозглашенной Донецкой народной ре...,Бывший СССР


In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12948 entries, 0 to 12947
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   12948 non-null  object
 1   text    12948 non-null  object
 2   topic   12948 non-null  object
dtypes: object(3)
memory usage: 303.6+ KB


In [14]:
data.topic.value_counts()

,count
topic,
Бывший СССР,1079
Дом,1079
Из жизни,1079
Интернет и СМИ,1079
Культура,1079
Мир,1079
Наука и техника,1079
Россия,1079
Силовые структуры,1079


Для банлансировки классов устанавливаем порог в 1000 значений, что позволило нам получить 14 классов и 1079 данных в каждом классе

In [15]:
morph = pymorphy3.MorphAnalyzer()

def clean_text(text):
    text = re.sub(r'[^А-Яа-яЁё\s]', '', text)
    text = re.sub(r'\s+', ' ', text)  # Удаление лишних пробелов
    text = text.strip().lower()  # Приведение к нижнему регистру
    return text

def lemmatize_text(text):
    words = text.split()  # Разбиваем текст на отдельные слова
    lemmatized_words = []

    for word in words:
        parsed_word = morph.parse(word)[0]  # Получаем анализатор для слова
        lemmatized_words.append(parsed_word.normal_form)  # Получаем лемму

    return ' '.join(lemmatized_words)

def normalize_text(text):
    cleaned_text = clean_text(text)
    lemmatized_text = lemmatize_text(cleaned_text)
    return lemmatized_text

In [16]:
data['cleaned_title'] = data['title'].apply(normalize_text)
data['cleaned_text'] = data['text'].apply(normalize_text)
data['cleaned_topic'] = data['topic'].apply(normalize_text)

In [17]:
data['full_text'] = data['cleaned_title'] + " " + data['cleaned_text']

In [18]:
label_encoder = LabelEncoder()
data['encoded_topic'] = label_encoder.fit_transform(data['cleaned_topic'])

In [19]:
df = data.drop(columns=['title', 'text', 'topic', 'cleaned_topic', 'cleaned_text', 'cleaned_title'])

In [20]:
train_data, temp_data = train_test_split(df, test_size=0.4, stratify=data['encoded_topic'], random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, stratify=temp_data['encoded_topic'], random_state=42)

In [21]:
corpus = [text.split() for text in train_data['full_text']]

w2v_model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=2, workers=4, sg=1, epochs=25)
w2v_model.save("word2vec_model.model")


In [22]:
print(w2v_model.wv.most_similar("глава", topn=5))
print(w2v_model.wv.doesnt_match(["страна", "республика"]))

[('руководитель', 0.7263447642326355), ('председатель', 0.6698890924453735), ('вожак', 0.6137674450874329), ('замглавы', 0.606938898563385), ('ермошин', 0.6044604182243347)]
республика


In [23]:
urllib.request.urlretrieve(
    "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
)

('ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz',
 <http.client.HTTPMessage at 0x7979a6b21c90>)

In [24]:
navec_path = 'navec_news_v1_1B_250K_300d_100q.tar'
navec_model = navec.Navec.load(navec_path)

rusvectores_path = 'ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz'
rusvectores_model = KeyedVectors.load_word2vec_format(rusvectores_path)

In [26]:
import spacy

nlp = spacy.load("ru_core_news_sm")

def add_pos_tags_spacy_pipe(texts):
    docs = nlp.pipe(texts)  # Обрабатываем тексты пачками
    return [" ".join([f"{token.text}_{token.pos_}" for token in doc]) for doc in docs]

# Применяем пакетную обработку
data['full_text_pos'] = add_pos_tags_spacy_pipe(data['full_text'])

In [27]:
df1 = data.drop(columns=['title', 'text', 'topic', 'cleaned_topic', 'cleaned_text', 'cleaned_title', 'full_text'])

In [28]:
train_data1, temp_data1 = train_test_split(df1, test_size=0.4, stratify=data['encoded_topic'], random_state=42)
val_data1, test_data1 = train_test_split(temp_data1, test_size=0.5, stratify=temp_data['encoded_topic'], random_state=42)

In [29]:
def get_average_vector(words, model):
    vectors = [model[word] for word in words if word in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

train_rusvectores = np.vstack(train_data1['full_text_pos'].apply(lambda x: get_average_vector(x.split(), rusvectores_model)))
val_rusvectores = np.vstack(val_data1['full_text_pos'].apply(lambda x: get_average_vector(x.split(), rusvectores_model)))

models = {
    "rusvectores": (train_rusvectores, val_rusvectores)
}

for name, (X_train, X_val) in models.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_data1['encoded_topic'])
    val_preds = clf.predict(X_val)
    acc = accuracy_score(val_data1['encoded_topic'], val_preds)
    print(f"Accuracy using {name}: {acc:.4f}")

Accuracy using rusvectores: 0.6892


In [30]:
def get_average_vector(words, model):
    vectors = [model[word] for word in words if word in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

train_w2v = np.vstack(train_data['full_text'].apply(lambda x: get_average_vector(x.split(), w2v_model.wv)))
val_w2v = np.vstack(val_data['full_text'].apply(lambda x: get_average_vector(x.split(), w2v_model.wv)))

train_navec = np.vstack(train_data['full_text'].apply(lambda x: get_average_vector(x.split(), navec_model)))
val_navec = np.vstack(val_data['full_text'].apply(lambda x: get_average_vector(x.split(), navec_model)))

#train_rusvectores = np.vstack(train_data['full_text'].apply(lambda x: get_average_vector(x.split(), rusvectores_model)))
#val_rusvectores = np.vstack(val_data['full_text'].apply(lambda x: get_average_vector(x.split(), rusvectores_model)))

# 10. Обучение и оценка моделей
models = {
    "w2v": (train_w2v, val_w2v),
    "navec": (train_navec, val_navec),
    #"rusvectores": (train_rusvectores, val_rusvectores)
}

for name, (X_train, X_val) in models.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_data['encoded_topic'])
    val_preds = clf.predict(X_val)
    acc = accuracy_score(val_data['encoded_topic'], val_preds)
    print(f"Accuracy using {name}: {acc:.4f}")


Accuracy using w2v: 0.7506
Accuracy using navec: 0.7625


Наибольший скор получился с обработкой navec

In [32]:
vectorizer = TfidfVectorizer()
vectorizer.fit(train_data['full_text'])

def get_tfidf_weighted_vector(text, model, vectorizer):
    words = text.split()
    word_vectors = []
    weights = []
    for word in words:
        if word in model and word in vectorizer.vocabulary_:
            word_vectors.append(model[word])
            weights.append(vectorizer.idf_[vectorizer.vocabulary_[word]])
    if word_vectors:
        return np.average(word_vectors, axis=0, weights=weights)
    else:
        return np.zeros(model.vector_size)

train_w2v = np.vstack(train_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, w2v_model.wv, vectorizer)))
val_w2v = np.vstack(val_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, w2v_model.wv, vectorizer)))

train_navec = np.vstack(train_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, navec_model, vectorizer)))
val_navec = np.vstack(val_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, navec_model, vectorizer)))

#train_rusvectores = np.vstack(train_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, rusvectores_model, vectorizer)))
#val_rusvectores = np.vstack(val_data['full_text'].apply(lambda x: get_tfidf_weighted_vector(x, rusvectores_model, vectorizer)))

models = {
    "w2v": (train_w2v, val_w2v),
    "navec": (train_navec, val_navec)
}

for name, (X_train, X_val) in models.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_data['encoded_topic'])
    val_preds = clf.predict(X_val)
    acc = accuracy_score(val_data['encoded_topic'], val_preds)
    print(f"Accuracy using {name} (TF-IDF weighted): {acc:.4f}")


Accuracy using w2v (TF-IDF weighted): 0.7537
Accuracy using navec (TF-IDF weighted): 0.7606


In [33]:
vectorizer = TfidfVectorizer()
vectorizer.fit(train_data1['full_text_pos'])

def get_tfidf_weighted_vector(text, model, vectorizer):
    words = text.split()
    word_vectors = []
    weights = []
    for word in words:
        if word in model and word in vectorizer.vocabulary_:
            word_vectors.append(model[word])
            weights.append(vectorizer.idf_[vectorizer.vocabulary_[word]])
    if word_vectors:
        return np.average(word_vectors, axis=0, weights=weights)
    else:
        return np.zeros(model.vector_size)

train_rusvectores = np.vstack(train_data1['full_text_pos'].apply(lambda x: get_tfidf_weighted_vector(x, rusvectores_model, vectorizer)))
val_rusvectores = np.vstack(val_data1['full_text_pos'].apply(lambda x: get_tfidf_weighted_vector(x, rusvectores_model, vectorizer)))

models = {
    "rusvectores": (train_rusvectores, val_rusvectores)
}

for name, (X_train, X_val) in models.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, train_data1['encoded_topic'])
    val_preds = clf.predict(X_val)
    acc = accuracy_score(val_data1['encoded_topic'], val_preds)
    print(f"Accuracy using {name} (TF-IDF weighted): {acc:.4f}")

Accuracy using rusvectores (TF-IDF weighted): 0.0830


После преобразования TF-IDF качество практически не поменялось, в обработке rusvectores качество снизилось